# **WECC 5-Cluster County Solar Validation**

This notebook checks the output from the WECC county-solar 5-cluster run.

1. Add county information to `gen_info.csv` so the generated UtilityPV rows can be traced back to counties.
2. Confirm that every county/county-region group gets at least one generated UtilityPV row.
3. Check whether groups with enough candidate solar rows are actually producing 5 clusters.
4. Inspect very small `gen_capacity_limit_mw` rows and test whether a minimum MW threshold may be useful.

The main outputs from this notebook are:

- `gen_info_wecc_county_solar_5clusters_with_county.csv`
- `county_region_5cluster_validation.csv`
- `capacity_threshold_summary.csv`
- `low_capacity_utilitypv_rows.csv`

In [94]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import re

PROJECT_ROOT = Path("/Users/laurenvo/Documents/Github/solar-county-analysis")
SWITCH_REPO = Path("/Users/laurenvo/Documents/Switch-USA-PG-ReEDS")

GEN_INFO_PATH = SWITCH_REPO / "switch/in/2030/s20x1_county_solar_5clusters/gen_info.csv"
RESOURCES_PATH = SWITCH_REPO / "pg/settings_wecc_county_solar_5clusters/resources.yml"
MODEL_DEF_PATH = SWITCH_REPO / "pg/settings_wecc_county_solar_5clusters/model_definition.yml"

SOLAR_META_PATH = SWITCH_REPO / "pg/extra_inputs/resource_groups_wecc_county_solar_test/ReEDS-cpas-patched/solar_lcoe_ReEDS_pg_schema_with_county_group.csv"

OUT_DIR = PROJECT_ROOT / "validation_outputs" / "wecc_county_solar_5clusters"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [GEN_INFO_PATH, RESOURCES_PATH, MODEL_DEF_PATH, SOLAR_META_PATH]:
    print(p.exists(), p)

True /Users/laurenvo/Documents/Switch-USA-PG-ReEDS/switch/in/2030/s20x1_county_solar_5clusters/gen_info.csv
True /Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/settings_wecc_county_solar_5clusters/resources.yml
True /Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/settings_wecc_county_solar_5clusters/model_definition.yml
True /Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/extra_inputs/resource_groups_wecc_county_solar_test/ReEDS-cpas-patched/solar_lcoe_ReEDS_pg_schema_with_county_group.csv


## Step 1: Load the 5-cluster output and source metadata

This section loads:

- the generated `gen_info.csv` from the 5-cluster WECC county-solar run,
- the `resources.yml` used for that run,
- the WECC model definition,
- and the county-enriched solar metadata.

The solar metadata is used as the source of truth for expected county-region groups and county names.

In [95]:
gen = pd.read_csv(GEN_INFO_PATH)
solar = pd.read_csv(SOLAR_META_PATH)

with open(MODEL_DEF_PATH, "r") as f:
    model_def = yaml.safe_load(f)

with open(RESOURCES_PATH, "r") as f:
    resources = yaml.safe_load(f)

print("gen_info rows:", len(gen))
print("gen_info columns:")
print(gen.columns.tolist())

print("\nsolar metadata rows:", len(solar))
print("solar metadata columns:")
print(solar.columns.tolist())

/var/folders/dg/bllrnyzj2fzf93_myxywvnnm0000gn/T/ipykernel_67104/3169245706.py:2: DtypeWarning: Columns (30) have mixed types. Specify dtype option on import or set low_memory=False.
  solar = pd.read_csv(SOLAR_META_PATH)


gen_info rows: 4059
gen_info columns:
['GENERATION_PROJECT', 'gen_connect_cost_per_mw', 'gen_amortization_period', 'gen_energy_source', 'gen_storage_efficiency', 'gen_capacity_limit_mw', 'gen_can_retire_early', 'gen_variable_om', 'gen_startup_fuel', 'gen_is_variable', 'gen_min_load_fraction', 'gen_tech', 'gen_load_zone', 'gen_max_age', 'gen_full_load_heat_rate', 'gen_ramp_limit_up', 'gen_ramp_limit_down', 'gen_min_uptime', 'gen_min_downtime', 'gen_startup_om', 'gen_self_discharge_rate', 'gen_is_baseload', 'gen_is_vpp', 'gen_max_annual_availability', 'gen_can_provide_spinning_reserves', 'gen_is_distributed', 'gen_scheduled_outage_rate', 'gen_forced_outage_rate']

solar metadata rows: 405737
solar metadata columns:
['Area', 'd_trans', 'd_sub', 'd_road', 'd_load_750', 'd_existing', 'd_plannedF', 'm_slope', 'm_popden', 'm_HMI', 'm_primeFarmland', 'incap', 'CPA_ID', 'm_aspect', 'm_aspect_min', 'Shape_Leng', 'm_landcover', 'exFacil', 'plFacil', 'Qual_Coal', 'Qual_Emp', 'Qual_Brown', 'anyQual

In [96]:
def clean_fips(x):
    """Convert FIPS-like values such as 4003, 4003.0, or '4003' into 5-digit strings."""
    if pd.isna(x):
        return pd.NA
    try:
        return str(int(float(x))).zfill(5)
    except Exception:
        s = str(x).strip()
        s = s.replace(".0", "")
        return s.zfill(5)


def find_col(df, possible_cols):
    for c in possible_cols:
        if c in df.columns:
            return c
    return None

## Step 2: Add county information to `gen_info.csv`

The generated UtilityPV resource names include county information in strings like:

`AZ2_utilitypv_class1_moderate_9_county_group_4003`

This section extracts:

- `model_region`
- `county_group`
- `county_name`
- `county_name_full`

and adds those columns back onto the generated `gen_info.csv`.

This makes it easier to trace each generated UtilityPV row back to the county it represents.

In [97]:
GEN_COL = find_col(gen, ["GENERATION_PROJECT", "gen_id", "generator", "gen_name"])
CAP_COL = find_col(gen, ["gen_capacity_limit_mw", "capacity_limit_mw", "capacity_mw"])

if GEN_COL is None:
    raise ValueError("Could not find generator name column. Check gen_info columns.")

if CAP_COL is None:
    raise ValueError("Could not find capacity limit column. Check gen_info columns.")

print("Using generator column:", GEN_COL)
print("Using capacity column:", CAP_COL)

gen2 = gen.copy()
gen2[GEN_COL] = gen2[GEN_COL].astype(str)

# Keep the county-clustered UtilityPV rows specifically
county_utilitypv_mask = gen2[GEN_COL].str.contains("utilitypv", case=False, na=False) & gen2[GEN_COL].str.contains("county_group_", case=False, na=False)

county_gen = gen2[county_utilitypv_mask].copy()

county_gen["county_group_raw"] = county_gen[GEN_COL].str.extract(r"county_group_([0-9]+)", expand=False)
county_gen["county_group"] = county_gen["county_group_raw"].apply(clean_fips)

# Example resource name starts with model region: AZ2_utilitypv...
county_gen["model_region"] = county_gen[GEN_COL].str.extract(r"^(.+?)_utilitypv", expand=False)

print("County UtilityPV rows:", len(county_gen))
print("Unique counties:", county_gen["county_group"].nunique())
print("Unique county-region pairs:", county_gen[["model_region", "county_group"]].drop_duplicates().shape[0])

county_gen[[GEN_COL, "model_region", "county_group", CAP_COL]].head()

Using generator column: GENERATION_PROJECT
Using capacity column: gen_capacity_limit_mw
County UtilityPV rows: 3514
Unique counties: 419
Unique county-region pairs: 739


,GENERATION_PROJECT,model_region,county_group,gen_capacity_limit_mw
313,AZ1_utilitypv_class1_moderate_0_county_group_4005,AZ1,04005,43153.8
314,AZ1_utilitypv_class1_moderate_1_county_group_4005,AZ1,04005,11439
315,AZ1_utilitypv_class1_moderate_2_county_group_4005,AZ1,04005,55576.1
316,AZ1_utilitypv_class1_moderate_3_county_group_4005,AZ1,04005,22497.2
317,AZ1_utilitypv_class1_moderate_4_county_group_4005,AZ1,04005,10486.6


In [98]:
# Add county names using Census county FIPS lookup
county_lookup_url = "https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt"

county_lookup = pd.read_csv(
    county_lookup_url,
    header=None,
    names=["state_abbrev", "state_fips", "county_fips", "county_name", "class_fips"],
    dtype={"state_fips": str, "county_fips": str}
)

county_lookup["county_group"] = (
    county_lookup["state_fips"].str.zfill(2)
    + county_lookup["county_fips"].str.zfill(3)
)

county_lookup["county_name_full"] = (
    county_lookup["county_name"] + ", " + county_lookup["state_abbrev"]
)

county_lookup = county_lookup[
    ["county_group", "county_name", "county_name_full", "state_abbrev"]
].drop_duplicates()

county_gen_named = county_gen.merge(
    county_lookup,
    on="county_group",
    how="left"
)

print("Rows with county name:", county_gen_named["county_name_full"].notna().sum())
print("Rows missing county name:", county_gen_named["county_name_full"].isna().sum())

county_gen_named[[GEN_COL, "model_region", "county_group", "county_name_full", CAP_COL]].head()

Rows with county name: 3509
Rows missing county name: 5


,GENERATION_PROJECT,model_region,county_group,county_name_full,gen_capacity_limit_mw
0,AZ1_utilitypv_class1_moderate_0_county_group_4005,AZ1,04005,"Coconino County, AZ",43153.8
1,AZ1_utilitypv_class1_moderate_1_county_group_4005,AZ1,04005,"Coconino County, AZ",11439
2,AZ1_utilitypv_class1_moderate_2_county_group_4005,AZ1,04005,"Coconino County, AZ",55576.1
3,AZ1_utilitypv_class1_moderate_3_county_group_4005,AZ1,04005,"Coconino County, AZ",22497.2
4,AZ1_utilitypv_class1_moderate_4_county_group_4005,AZ1,04005,"Coconino County, AZ",10486.6


In [99]:
print("Rows with county name:", county_gen_named["county_name_full"].notna().sum())
print("Rows missing county name:", county_gen_named["county_name_full"].isna().sum())

Rows with county name: 3509
Rows missing county name: 5


In [100]:
missing_name_rows = county_gen_named[
    county_gen_named["county_name_full"].isna()
].copy()

missing_name_cols = [
    GEN_COL,
    "model_region",
    "county_group",
    "county_group_raw",
    CAP_COL,
]

print("Missing county-name rows:")
display(missing_name_rows[missing_name_cols].drop_duplicates())

missing_name_rows[missing_name_cols].drop_duplicates().to_csv(
    OUT_DIR / "missing_county_name_rows.csv",
    index=False
)

Missing county-name rows:


,GENERATION_PROJECT,model_region,county_group,county_group_raw,gen_capacity_limit_mw
2464,SD1_utilitypv_class1_moderate_75_county_group_...,SD1,46102,46102,3036.6
2465,SD1_utilitypv_class1_moderate_76_county_group_...,SD1,46102,46102,14131.2
2466,SD1_utilitypv_class1_moderate_77_county_group_...,SD1,46102,46102,9915.2
2467,SD1_utilitypv_class1_moderate_78_county_group_...,SD1,46102,46102,5253.7
2468,SD1_utilitypv_class1_moderate_79_county_group_...,SD1,46102,46102,9447.6


In [101]:
# Manual display-name patch for county FIPS that did not resolve in Census lookup.
# This is only for traceability/reporting, not a modeling change.
manual_county_lookup = pd.DataFrame([
    {
        "county_group": "46102",
        "county_name": "Oglala Lakota County",
        "county_name_full": "Oglala Lakota County, SD",
        "state_abbrev": "SD",
    }
])

county_lookup = pd.concat(
    [county_lookup, manual_county_lookup],
    ignore_index=True
).drop_duplicates(subset=["county_group"], keep="last")

county_gen_named = county_gen.drop(
    columns=["county_name", "county_name_full", "state_abbrev"],
    errors="ignore"
).merge(
    county_lookup,
    on="county_group",
    how="left"
)

print("Rows missing county name after manual patch:", county_gen_named["county_name_full"].isna().sum())

display(
    county_gen_named[
        county_gen_named["county_group"].eq("46102")
    ][[GEN_COL, "model_region", "county_group", "county_name_full", CAP_COL]]
)

Rows missing county name after manual patch: 0


,GENERATION_PROJECT,model_region,county_group,county_name_full,gen_capacity_limit_mw
2464,SD1_utilitypv_class1_moderate_75_county_group_...,SD1,46102,"Oglala Lakota County, SD",3036.6
2465,SD1_utilitypv_class1_moderate_76_county_group_...,SD1,46102,"Oglala Lakota County, SD",14131.2
2466,SD1_utilitypv_class1_moderate_77_county_group_...,SD1,46102,"Oglala Lakota County, SD",9915.2
2467,SD1_utilitypv_class1_moderate_78_county_group_...,SD1,46102,"Oglala Lakota County, SD",5253.7
2468,SD1_utilitypv_class1_moderate_79_county_group_...,SD1,46102,"Oglala Lakota County, SD",9447.6


## Step 3: Validate county-region cluster counts

This section checks whether the 5-cluster run behaved as expected.

For each `(model_region, county_group)` pair:

- If there are fewer than 5 candidate solar rows, PowerGenome can only create up to that number.
- If there are at least 5 candidate solar rows, we expect 5 generated clusters.

So the expected cluster count is:

`min(candidate_rows, 5)`

This section compares the expected count from the solar metadata against the actual count in `gen_info.csv`.

In [102]:
# Map p-regions to WECC model regions
p_to_model_region = {}

for model_region, p_regions in model_def["region_aggregations"].items():
    for p in p_regions:
        p_to_model_region[str(p)] = model_region

solar_wecc = solar.copy()
solar_wecc["ipm_region"] = solar_wecc["ipm_region"].astype(str)
solar_wecc["model_region"] = solar_wecc["ipm_region"].map(p_to_model_region)
solar_wecc["county_group"] = solar_wecc["county_group"].apply(clean_fips)

solar_wecc = solar_wecc.dropna(subset=["model_region", "county_group"]).copy()

expected = (
    solar_wecc.groupby(["model_region", "county_group"])
    .agg(candidate_rows=("county_group", "size"))
    .reset_index()
)

expected["expected_clusters_n5"] = expected["candidate_rows"].clip(upper=5)

expected = expected.merge(
    county_lookup[["county_group", "county_name", "county_name_full", "state_abbrev"]],
    on="county_group",
    how="left"
)

print("Expected county-region groups:", len(expected))
print("Expected clusters total:", expected["expected_clusters_n5"].sum())
print("Groups with fewer than 5 candidate rows:", (expected["candidate_rows"] < 5).sum())
print("Groups with at least 5 candidate rows:", (expected["candidate_rows"] >= 5).sum())
print("Expected rows missing county name:", expected["county_name_full"].isna().sum())

display(expected.head())

Expected county-region groups: 739
Expected clusters total: 3516
Groups with fewer than 5 candidate rows: 64
Groups with at least 5 candidate rows: 675
Expected rows missing county name: 0


,model_region,county_group,candidate_rows,expected_clusters_n5,county_name,county_name_full,state_abbrev
0,AZ1,04005,1214,5,Coconino County,"Coconino County, AZ",AZ
1,AZ1,04012,377,5,La Paz County,"La Paz County, AZ",AZ
2,AZ1,04013,51,5,Maricopa County,"Maricopa County, AZ",AZ
3,AZ1,04015,981,5,Mohave County,"Mohave County, AZ",AZ
4,AZ1,04025,497,5,Yavapai County,"Yavapai County, AZ",AZ


In [103]:
actual = (
    county_gen_named.groupby(["model_region", "county_group"])
    .size()
    .reset_index(name="actual_clusters")
)

qa = expected.merge(
    actual,
    on=["model_region", "county_group"],
    how="left"
)

qa["actual_clusters"] = qa["actual_clusters"].fillna(0).astype(int)

qa["has_at_least_one"] = qa["actual_clusters"] >= 1
qa["matches_expected_min_candidate_5"] = qa["actual_clusters"] == qa["expected_clusters_n5"]

qa["should_have_5"] = qa["candidate_rows"] >= 5
qa["has_5_when_possible"] = np.where(
    qa["should_have_5"],
    qa["actual_clusters"] == 5,
    True
)

print("===== QA SUMMARY =====")
print("Expected county-region groups:", len(qa))
print("Actual county-region groups:", actual.shape[0])
print("Groups with at least 1 actual cluster:", qa["has_at_least_one"].sum())
print("Missing groups:", (~qa["has_at_least_one"]).sum())
print("Groups that match min(candidate_rows, 5):", qa["matches_expected_min_candidate_5"].sum())
print("Groups that do NOT match expected:", (~qa["matches_expected_min_candidate_5"]).sum())
print("Groups with >=5 candidates:", qa["should_have_5"].sum())
print("Groups with >=5 candidates that produced exactly 5:", qa.loc[qa["should_have_5"], "has_5_when_possible"].sum())

qa.sort_values(["matches_expected_min_candidate_5", "candidate_rows"]).head(20)

===== QA SUMMARY =====
Expected county-region groups: 739
Actual county-region groups: 739
Groups with at least 1 actual cluster: 739
Missing groups: 0
Groups that match min(candidate_rows, 5): 737
Groups that do NOT match expected: 2
Groups with >=5 candidates: 675
Groups with >=5 candidates that produced exactly 5: 674


,model_region,county_group,candidate_rows,expected_clusters_n5,county_name,county_name_full,state_abbrev,actual_clusters,has_at_least_one,matches_expected_min_candidate_5,should_have_5,has_5_when_possible
386,NV1,06017,3,3,El Dorado County,"El Dorado County, CA",CA,2,True,False,False,True
375,NM1,35045,5,5,San Juan County,"San Juan County, NM",NM,4,True,False,True,False
29,AZ3,04007,1,1,Gila County,"Gila County, AZ",AZ,1,True,True,False,True
42,AZ4,04005,1,1,Coconino County,"Coconino County, AZ",AZ,1,True,True,False,True
61,CA1,41029,1,1,Jackson County,"Jackson County, OR",OR,1,True,True,False,True
77,CA3,06059,1,1,Orange County,"Orange County, CA",CA,1,True,True,False,True
81,CA4,06003,1,1,Alpine County,"Alpine County, CA",CA,1,True,True,False,True
128,CO1,08015,1,1,Chaffee County,"Chaffee County, CO",CO,1,True,True,False,True
138,CO1,08051,1,1,Gunnison County,"Gunnison County, CO",CO,1,True,True,False,True
167,CO2,08033,1,1,Dolores County,"Dolores County, CO",CO,1,True,True,False,True


In [104]:
# Export county-region groups where actual cluster count does not match min(candidate_rows, 5)

mismatch = qa[~qa["matches_expected_min_candidate_5"]].copy()

mismatch_out = OUT_DIR / "cluster_count_mismatches.csv"
mismatch.to_csv(mismatch_out, index=False)

print("Wrote:", mismatch_out)
print("Number of mismatch groups:", len(mismatch))

display(mismatch)

Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/cluster_count_mismatches.csv
Number of mismatch groups: 2


,model_region,county_group,candidate_rows,expected_clusters_n5,county_name,county_name_full,state_abbrev,actual_clusters,has_at_least_one,matches_expected_min_candidate_5,should_have_5,has_5_when_possible
375,NM1,35045,5,5,San Juan County,"San Juan County, NM",NM,4,True,False,True,False
386,NV1,06017,3,3,El Dorado County,"El Dorado County, CA",CA,2,True,False,False,True


In [105]:
# Export the candidate rows and generated gen_info rows for the mismatch groups

candidate_detail_rows = []
actual_detail_rows = []

for _, row in mismatch.iterrows():
    mr = row["model_region"]
    cg = row["county_group"]

    cand = solar_wecc[
        (solar_wecc["model_region"] == mr)
        & (solar_wecc["county_group"] == cg)
    ].copy()
    cand["mismatch_model_region"] = mr
    cand["mismatch_county_group"] = cg
    candidate_detail_rows.append(cand)

    actual = county_gen_named[
        (county_gen_named["model_region"] == mr)
        & (county_gen_named["county_group"] == cg)
    ].copy()
    actual_detail_rows.append(actual)

if candidate_detail_rows:
    candidate_detail = pd.concat(candidate_detail_rows, ignore_index=True)
else:
    candidate_detail = pd.DataFrame()

if actual_detail_rows:
    actual_detail = pd.concat(actual_detail_rows, ignore_index=True)
else:
    actual_detail = pd.DataFrame()

candidate_detail_out = OUT_DIR / "cluster_mismatch_candidate_rows.csv"
actual_detail_out = OUT_DIR / "cluster_mismatch_actual_gen_info_rows.csv"

candidate_detail.to_csv(candidate_detail_out, index=False)
actual_detail.to_csv(actual_detail_out, index=False)

print("Wrote:", candidate_detail_out)
print("Wrote:", actual_detail_out)

display_cols = [
    GEN_COL,
    "model_region",
    "county_group",
    "county_name_full",
    CAP_COL,
]

display_cols = [c for c in display_cols if c in actual_detail.columns]

display(actual_detail[display_cols])

Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/cluster_mismatch_candidate_rows.csv
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/cluster_mismatch_actual_gen_info_rows.csv


,GENERATION_PROJECT,model_region,county_group,county_name_full,gen_capacity_limit_mw
0,NM1_utilitypv_class1_moderate_110_county_group...,NM1,35045,"San Juan County, NM",71.7
1,NM1_utilitypv_class1_moderate_111_county_group...,NM1,35045,"San Juan County, NM",25.7
2,NM1_utilitypv_class1_moderate_112_county_group...,NM1,35045,"San Juan County, NM",51.1
3,NM1_utilitypv_class1_moderate_113_county_group...,NM1,35045,"San Juan County, NM",97.9
4,NV1_utilitypv_class1_moderate_5_county_group_6017,NV1,06017,"El Dorado County, CA",29.2
5,NV1_utilitypv_class1_moderate_6_county_group_6017,NV1,06017,"El Dorado County, CA",36


## Step 4: Inspect small `gen_capacity_limit_mw` rows

This section checks:

- How many generated county UtilityPV rows have very small `gen_capacity_limit_mw`.
- Which counties/resources are affected.
- What would happen if we dropped resources below different minimum MW thresholds.

The key concern is whether applying a minimum threshold would remove county coverage.

In [106]:
county_gen_named[CAP_COL] = pd.to_numeric(county_gen_named[CAP_COL], errors="coerce")

capacity_summary = county_gen_named[CAP_COL].describe()
capacity_summary

count     3514.000000
mean      3334.000285
std       5837.624771
min          1.800000
25%        235.025000
50%       1161.200000
75%       3984.725000
max      76006.700000
Name: gen_capacity_limit_mw, dtype: float64

In [107]:
low_capacity_rows = county_gen_named.sort_values(CAP_COL).copy()

cols_to_show = [GEN_COL, "model_region", "county_group", "county_name_full", CAP_COL]
low_capacity_rows[cols_to_show].head(30)

,GENERATION_PROJECT,model_region,county_group,county_name_full,gen_capacity_limit_mw
66,AZ2_utilitypv_class1_moderate_9_county_group_4003,AZ2,04003,"Cochise County, AZ",1.8
3090,WA4_utilitypv_class1_moderate_47_county_group_...,WA4,53005,"Benton County, WA",2.4
1740,NM1_utilitypv_class1_moderate_67_county_group_...,NM1,35013,"Dona Ana County, NM",3.3
2723,UT2_utilitypv_class1_moderate_51_county_group_...,UT2,49051,"Wasatch County, UT",4.3
1744,NM1_utilitypv_class1_moderate_71_county_group_...,NM1,35017,"Grant County, NM",4.5
657,CO1_utilitypv_class1_moderate_62_county_group_...,CO1,08059,"Jefferson County, CO",4.5
534,CA4_utilitypv_class1_moderate_155_county_group...,CA4,06087,"Santa Cruz County, CA",4.5
2105,OR1_utilitypv_class1_moderate_7_county_group_6023,OR1,06023,"Humboldt County, CA",4.5
538,CA4_utilitypv_class1_moderate_159_county_group...,CA4,06089,"Shasta County, CA",4.5
715,CO1_utilitypv_class1_moderate_120_county_group...,CO1,08117,"Summit County, CO",4.5


In [108]:
target_county = "04003"

target = county_gen_named[county_gen_named["county_group"] == target_county].sort_values(CAP_COL)

print("Rows for county", target_county, ":", len(target))
target[cols_to_show]

Rows for county 04003 : 10


,GENERATION_PROJECT,model_region,county_group,county_name_full,gen_capacity_limit_mw
66,AZ2_utilitypv_class1_moderate_9_county_group_4003,AZ2,04003,"Cochise County, AZ",1.8
64,AZ2_utilitypv_class1_moderate_7_county_group_4003,AZ2,04003,"Cochise County, AZ",6.8
65,AZ2_utilitypv_class1_moderate_8_county_group_4003,AZ2,04003,"Cochise County, AZ",7.9
63,AZ2_utilitypv_class1_moderate_6_county_group_4003,AZ2,04003,"Cochise County, AZ",83.4
62,AZ2_utilitypv_class1_moderate_5_county_group_4003,AZ2,04003,"Cochise County, AZ",222.0
202,AZ4_utilitypv_class1_moderate_4_county_group_4003,AZ4,04003,"Cochise County, AZ",5269.4
201,AZ4_utilitypv_class1_moderate_3_county_group_4003,AZ4,04003,"Cochise County, AZ",8624.9
200,AZ4_utilitypv_class1_moderate_2_county_group_4003,AZ4,04003,"Cochise County, AZ",17118.1
199,AZ4_utilitypv_class1_moderate_1_county_group_4003,AZ4,04003,"Cochise County, AZ",27113.1
198,AZ4_utilitypv_class1_moderate_0_county_group_4003,AZ4,04003,"Cochise County, AZ",28889.3


In [109]:
thresholds = [0, 1, 2, 5, 10, 25, 50, 100]

threshold_rows = []

all_expected_groups = set(zip(qa["model_region"], qa["county_group"]))
all_expected_counties = set(qa["county_group"])

for threshold in thresholds:
    kept = county_gen_named[county_gen_named[CAP_COL] >= threshold].copy()
    
    kept_groups = set(zip(kept["model_region"], kept["county_group"]))
    kept_counties = set(kept["county_group"])
    
    missing_groups = all_expected_groups - kept_groups
    missing_counties = all_expected_counties - kept_counties
    
    threshold_rows.append({
        "threshold_mw": threshold,
        "rows_kept": len(kept),
        "rows_dropped": len(county_gen_named) - len(kept),
        "county_region_groups_kept": len(kept_groups),
        "county_region_groups_missing": len(missing_groups),
        "unique_counties_kept": len(kept_counties),
        "unique_counties_missing": len(missing_counties),
        "total_capacity_mw_kept": kept[CAP_COL].sum(),
        "total_capacity_mw_dropped": county_gen_named[CAP_COL].sum() - kept[CAP_COL].sum(),
    })

threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary

,threshold_mw,rows_kept,rows_dropped,county_region_groups_kept,county_region_groups_missing,unique_counties_kept,unique_counties_missing,total_capacity_mw_kept,total_capacity_mw_dropped
0,0,3514,0,739,0,419,0,11715677.0,0.0
1,1,3514,0,739,0,419,0,11715677.0,0.0
2,2,3513,1,739,0,419,0,11715675.2,1.8
3,5,3503,11,739,0,419,0,11715633.4,43.6
4,10,3481,33,738,1,419,0,11715482.1,194.9
5,25,3369,145,729,10,418,1,11713276.3,2400.7
6,50,3195,319,707,32,416,3,11706847.4,8829.6
7,100,2991,523,679,60,411,8,11691795.4,23881.6


## Step 5: Check whether the repo already has an obvious minimum threshold

This section searches the local settings and PowerGenome files for terms related to minimum capacity thresholds.

This does not prove that no threshold exists anywhere, but it helps check whether there is an obvious setting in the current resources/settings files.

In [110]:
search_terms = [
    "gen_capacity_limit_mw",
    "capacity_limit",
    "min_capacity",
    "minimum",
    "threshold",
    "mw_per_bin",
]

search_dirs = [
    SWITCH_REPO / "pg",
    SWITCH_REPO / "PowerGenome" / "powergenome",
]

matches = []

for search_dir in search_dirs:
    if not search_dir.exists():
        continue
    
    for path in search_dir.rglob("*"):
        if path.suffix.lower() not in [".yml", ".yaml", ".py", ".csv"]:
            continue
        
        try:
            text = path.read_text(errors="ignore")
        except Exception:
            continue
        
        for i, line in enumerate(text.splitlines(), start=1):
            lower = line.lower()
            if any(term.lower() in lower for term in search_terms):
                matches.append({
                    "file": str(path.relative_to(SWITCH_REPO)),
                    "line": i,
                    "text": line.strip()
                })

matches_df = pd.DataFrame(matches)
print("Matches found:", len(matches_df))
matches_df.head(100)

Matches found: 86


,file,line,text
0,pg/settings/scenario_management.yml,182,--exclude-modules study_modules.min_capacity_c...
1,pg/settings/extra_inputs.yml,18,capacity_limit_spur_fn: resource_capacity_spur...
2,pg/settings/transmission.yml,147,# (allows the minimum of the two possible valu...
3,pg/settings_wecc_county_solar_test/scenario_ma...,182,--exclude-modules study_modules.min_capacity_c...
4,pg/settings_wecc_county_solar_test/extra_input...,18,capacity_limit_spur_fn: resource_capacity_spur...
...,...,...,...
81,PowerGenome/powergenome/cluster/renewables.py,635,from available site capacity and `mw_per_bin`/...
82,PowerGenome/powergenome/cluster/renewables.py,637,"if ""mw_per_bin"" in b:"
83,PowerGenome/powergenome/cluster/renewables.py,639,"logger.warning(""Overwriting 'bins' based on mw..."
84,PowerGenome/powergenome/cluster/renewables.py,640,"b[""bins""] = max(int(data[""mw""].sum() / b[""mw_p..."


## Step 6: Save outputs

This section saves the updated `gen_info.csv` and validation tables.

The main files to send/review are:

- `gen_info_wecc_county_solar_5clusters_with_county.csv`
- `county_region_5cluster_validation.csv`
- `capacity_threshold_summary.csv`
- `low_capacity_utilitypv_rows.csv`
- `threshold_setting_search_matches.csv`

In [111]:
# Final export cell

from pathlib import Path
import pandas as pd
import shutil

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

written_files = []
missing_objects = []

def save_df_from_possible_names(possible_names, filename, description):
    """
    Save the first DataFrame found from a list of possible variable names.
    This makes the final cell robust if earlier cells used slightly different names.
    """
    for name in possible_names:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame):
            out_path = OUT_DIR / filename
            obj.to_csv(out_path, index=False)
            written_files.append({
                "file": filename,
                "rows": len(obj),
                "description": description,
            })
            print(f"Wrote: {out_path} ({len(obj):,} rows)")
            return out_path

    missing_objects.append((filename, possible_names))
    print(f"Skipped: {filename} — none of these DataFrame variables exist: {possible_names}")
    return None


# gen_info with county columns
save_df_from_possible_names(
    ["county_gen_named"],
    "gen_info_wecc_county_solar_5clusters_with_county.csv",
    "Generated gen_info rows with county_group and readable county name columns added.",
)

# Main validation table
save_df_from_possible_names(
    ["qa"],
    "county_region_5cluster_validation.csv",
    "Expected-vs-actual county-region cluster validation table.",
)

# County-name issue proof / patch record
save_df_from_possible_names(
    ["missing_name_rows", "missing_county_name_rows"],
    "missing_county_name_rows.csv",
    "Rows that were missing county names before the manual display-name patch.",
)

# Cluster-count mismatch files
save_df_from_possible_names(
    ["mismatch"],
    "cluster_count_mismatches.csv",
    "County-region groups where actual cluster count did not match min(candidate rows, 5).",
)

save_df_from_possible_names(
    ["candidate_detail", "candidate_detail_rows_df"],
    "cluster_mismatch_candidate_rows.csv",
    "Original candidate solar rows for the mismatch county-region groups.",
)

save_df_from_possible_names(
    ["actual_detail", "actual_detail_rows_df"],
    "cluster_mismatch_actual_gen_info_rows.csv",
    "Generated gen_info rows for the mismatch county-region groups.",
)

# Capacity threshold files
save_df_from_possible_names(
    ["threshold_summary"],
    "capacity_threshold_summary.csv",
    "Summary of how different gen_capacity_limit_mw thresholds affect retained rows, county-region groups, and counties.",
)

save_df_from_possible_names(
    ["low_capacity_utilitypv_rows", "low_capacity_rows", "low_capacity"],
    "low_capacity_utilitypv_rows.csv",
    "Generated UtilityPV rows with low gen_capacity_limit_mw values.",
)

# Optional search result showing possible threshold settings in yml files
save_df_from_possible_names(
    ["threshold_setting_search_matches", "setting_matches", "threshold_matches"],
    "threshold_setting_search_matches.csv",
    "Search results for existing minimum-capacity or threshold-related settings in resources/settings files.",
)


# Optional: copy resources.yml if a path variable exists or if file is found nearby
resource_output_name = "resources_wecc_county_solar_5clusters.yml"

possible_resource_vars = [
    "RESOURCES_PATH",
    "RESOURCES_YML",
    "resources_path",
    "resources_yml",
    "resources_file",
]

copied_resource = False

for var_name in possible_resource_vars:
    p = globals().get(var_name)
    if p is not None:
        p = Path(p)
        if p.exists():
            out_path = OUT_DIR / resource_output_name
            shutil.copy2(p, out_path)
            written_files.append({
                "file": resource_output_name,
                "rows": "",
                "description": "resources.yml file used for the 5-cluster county-solar run.",
            })
            print(f"Wrote: {out_path}")
            copied_resource = True
            break

if not copied_resource:
    # Fallback: check if the resources file was already manually copied into OUT_DIR
    existing_resource = OUT_DIR / resource_output_name
    if existing_resource.exists():
        written_files.append({
            "file": resource_output_name,
            "rows": "",
            "description": "resources.yml file used for the 5-cluster county-solar run.",
        })
        print(f"Found existing: {existing_resource}")
        copied_resource = True
    else:
        print(
            f"Note: did not copy {resource_output_name}. "
            "If Jenny needs it, manually copy the resources.yml into OUT_DIR."
        )

manifest = pd.DataFrame(written_files)
manifest_path = OUT_DIR / "files_to_send_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Wrote: {manifest_path}")

# Create a short README summary
readme_path = OUT_DIR / "README_for_Jenny.md"

summary_lines = [
    "# WECC County-Solar 5-Cluster Validation Outputs",
    "",
    "## Main result",
    "- The generated output preserves all expected county-region groups.",
    "- The county-name lookup issue was a display/traceability issue, not a missing-cluster issue.",
    "- A 5 MW capacity threshold appears to be the safest tested threshold if we want to preserve every county-region group.",
    "",
    "## Files included",
]

for item in written_files:
    row_text = f"- `{item['file']}`"
    if item["rows"] != "":
        row_text += f" — {item['rows']:,} rows"
    row_text += f": {item['description']}"
    summary_lines.append(row_text)

if missing_objects:
    summary_lines += [
        "",
        "## Files not written automatically",
    ]
    for filename, names in missing_objects:
        summary_lines.append(
            f"- `{filename}` was not written because none of these variables existed: {names}"
        )

readme_path.write_text("\n".join(summary_lines))
print(f"Wrote: {readme_path}")

print("\nFinal files:")
display(manifest)

Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/gen_info_wecc_county_solar_5clusters_with_county.csv (3,514 rows)
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/county_region_5cluster_validation.csv (739 rows)
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/missing_county_name_rows.csv (5 rows)
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/cluster_count_mismatches.csv (2 rows)
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/cluster_mismatch_candidate_rows.csv (8 rows)
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/cluster_mismatch_actual_gen_info_rows.csv (6 rows)
Wrote: /Users/laurenvo/Documents/Github/solar-county-analys

,file,rows,description
0,gen_info_wecc_county_solar_5clusters_with_coun...,3514,Generated gen_info rows with county_group and ...
1,county_region_5cluster_validation.csv,739,Expected-vs-actual county-region cluster valid...
2,missing_county_name_rows.csv,5,Rows that were missing county names before the...
3,cluster_count_mismatches.csv,2,County-region groups where actual cluster coun...
4,cluster_mismatch_candidate_rows.csv,8,Original candidate solar rows for the mismatch...
5,cluster_mismatch_actual_gen_info_rows.csv,6,Generated gen_info rows for the mismatch count...
6,capacity_threshold_summary.csv,8,Summary of how different gen_capacity_limit_mw...
7,low_capacity_utilitypv_rows.csv,3514,Generated UtilityPV rows with low gen_capacity...
8,resources_wecc_county_solar_5clusters.yml,,resources.yml file used for the 5-cluster coun...


In [112]:
# Small cleanup after final export cell

from pathlib import Path
import pandas as pd

OUT_DIR = Path(OUT_DIR)

# 1. Save threshold/settings search matches correctly
if "matches_df" in globals() and isinstance(matches_df, pd.DataFrame):
    threshold_matches_out = OUT_DIR / "threshold_setting_search_matches.csv"
    matches_df.to_csv(threshold_matches_out, index=False)
    print("Wrote:", threshold_matches_out, f"({len(matches_df):,} rows)")
else:
    print("matches_df not found; threshold_setting_search_matches.csv not written.")

# 2. Save true low-capacity subsets, not the full sorted table
low_capacity_below_5 = county_gen_named[county_gen_named[CAP_COL] < 5].sort_values(CAP_COL).copy()
low_capacity_below_10 = county_gen_named[county_gen_named[CAP_COL] < 10].sort_values(CAP_COL).copy()

below_5_out = OUT_DIR / "low_capacity_utilitypv_rows_below_5mw.csv"
below_10_out = OUT_DIR / "low_capacity_utilitypv_rows_below_10mw.csv"

low_capacity_below_5.to_csv(below_5_out, index=False)
low_capacity_below_10.to_csv(below_10_out, index=False)

print("Wrote:", below_5_out, f"({len(low_capacity_below_5):,} rows)")
print("Wrote:", below_10_out, f"({len(low_capacity_below_10):,} rows)")

# 3. Rewrite manifest with the corrected extra files
manifest_extra = pd.DataFrame([
    {
        "file": "threshold_setting_search_matches.csv",
        "rows": len(matches_df) if "matches_df" in globals() and isinstance(matches_df, pd.DataFrame) else "",
        "description": "Search results for existing minimum-capacity or threshold-related settings in resources/settings files.",
    },
    {
        "file": "low_capacity_utilitypv_rows_below_5mw.csv",
        "rows": len(low_capacity_below_5),
        "description": "Generated UtilityPV rows with gen_capacity_limit_mw below 5 MW.",
    },
    {
        "file": "low_capacity_utilitypv_rows_below_10mw.csv",
        "rows": len(low_capacity_below_10),
        "description": "Generated UtilityPV rows with gen_capacity_limit_mw below 10 MW.",
    },
])

manifest_path = OUT_DIR / "files_to_send_manifest.csv"
old_manifest = pd.read_csv(manifest_path)

new_manifest = pd.concat([old_manifest, manifest_extra], ignore_index=True)
new_manifest = new_manifest.drop_duplicates(subset=["file"], keep="last")

new_manifest.to_csv(manifest_path, index=False)
print("Updated:", manifest_path)

display(new_manifest)

Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/threshold_setting_search_matches.csv (86 rows)
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/low_capacity_utilitypv_rows_below_5mw.csv (11 rows)
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/low_capacity_utilitypv_rows_below_10mw.csv (33 rows)
Updated: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/files_to_send_manifest.csv


,file,rows,description
0,gen_info_wecc_county_solar_5clusters_with_coun...,3514.0,Generated gen_info rows with county_group and ...
1,county_region_5cluster_validation.csv,739.0,Expected-vs-actual county-region cluster valid...
2,missing_county_name_rows.csv,5.0,Rows that were missing county names before the...
3,cluster_count_mismatches.csv,2.0,County-region groups where actual cluster coun...
4,cluster_mismatch_candidate_rows.csv,8.0,Original candidate solar rows for the mismatch...
5,cluster_mismatch_actual_gen_info_rows.csv,6.0,Generated gen_info rows for the mismatch count...
6,capacity_threshold_summary.csv,8.0,Summary of how different gen_capacity_limit_mw...
7,low_capacity_utilitypv_rows.csv,3514.0,Generated UtilityPV rows with low gen_capacity...
8,resources_wecc_county_solar_5clusters.yml,NaN,resources.yml file used for the 5-cluster coun...
9,threshold_setting_search_matches.csv,86.0,Search results for existing minimum-capacity o...
